<div align='center'>

# Practica 5

<img src='https://media2.giphy.com/media/v1.Y2lkPTc5MGI3NjExaHhlMTBkeWt4NGtjcTJ3Znc1MTlueDI0dm1mOG1tZ2t6ODNmYmw5aSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/uHox9Jm5TyTPa/giphy.gif'>

</div>

## Creamos el entorno virtual
Vamos a tener que instalar el jdk y configurar las variables de entorno


In [3]:
from pyspark.sql import SparkSession
import os, sys

# Ruta del JDK (ajustá si cambia)
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Fuerza a Spark a usar el mismo Python del venv
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


## Ejercicio 1

Dado el siguiente **RDD** almacenado en la variable `rdd`:

| Partition 1 | Partition 2 | Partition 3 | Partition 4 |
| :---------: | :---------: | :---------: | :---------: |
|    34  21   |    23  45   |    3  21    |    30  91   |
|    21  34   |    12  12   |    15  10   |    31  32   |
|    10  18   |    36  18   |    14  18   |    32  53   |
|    32  45   |    4  97    |    3  15    |    19  35   |

---

### Pregunta

Responda: ¿Qué imprime cada uno de los siguientes scripts (sin ejecutarlo)?

In [4]:
spark = SparkSession.builder \
    .appName("RDD-Ejercicio01") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

data = [
    (34, 21), (21, 34), (10, 18), (32, 45),  # Partición 1
    (23, 45), (12, 12), (36, 18), (4, 97),   # Partición 2
    (3, 21), (15, 10), (14, 18), (3, 15),    # Partición 3
    (30, 91), (31, 32), (32, 53), (19, 35)   # Partición 4
]

rdd = sc.parallelize(data, 4)

### a)

```python
res = rdd.map(lambda t: (t[0] + t[1]) * 2)
print(res.first())
```

Ese código toma cada par de números del RDD y calcula la **suma de los dos, multiplicada por dos**.
Después, muestra solo el **primer resultado** de todos esos cálculos.

En otras palabras:

> “Por cada tupla `(a, b)` del conjunto de datos, hace `(a + b) * 2`,
> y luego imprime el primer valor obtenido.”

Por ejemplo, con la primera tupla `(34, 21)` → `(34 + 21) * 2 = 110`.
Por eso, el programa imprime **110**.


In [5]:
res = rdd.map(lambda t: (t[0] + t[1]) * 2)
print("Resultado del punto (a):", res.first())

Resultado del punto (a): 110


### b)

```python
res = rdd.filter(lambda t: t[0] >= t[1])
print(res.take(3))
```

Ese código revisa todas las parejas de números del conjunto y **se queda solo con aquellas en las que el primer número es mayor o igual que el segundo**.
Después, **muestra las tres primeras parejas** que cumplen esa condición.


In [6]:
res = rdd.filter(lambda t: t[0] >= t[1])
print(res.take(3))

[(34, 21), (12, 12), (36, 18)]


### c)

```python
res = rdd.map(lambda t: (t[0], t[1], t[0] / t[1]))
res = res.filter(lambda t: t[2] < 0.5)
res = res.reduce(lambda t1, t2: t1 if t1[2] < t2[2] else t2)

print(res)
```

Primero, el código **crea una nueva lista** donde cada par de números incluye también el **resultado de dividir el primero por el segundo**.
Después, **filtra** esa lista quedándose solo con los pares donde el resultado de la división es **menor que 0.5**.
Por último, **compara todos esos casos** y se queda con el que tenga **la división más chica de todos**.

En este conjunto de datos, esa tupla resulta ser **(4, 97, 0.041)**.


In [7]:
res = rdd.map(lambda t: (t[0], t[1], t[0] / t[1]))
res = res.filter(lambda t: t[2] < 0.5)
res = res.reduce(lambda t1, t2: t1 if t1[2] < t2[2] else t2)
print(res)

(4, 97, 0.041237113402061855)


### d)

```python
r1 = rdd.map(lambda t: t[0])
r2 = rdd.map(lambda t: t[1])
r1 = r1.distinct()
r2 = r2.distinct()
res = r2.union(r1)
print(res.collect())
```

Ese código **separa** todos los primeros números del conjunto en una lista y todos los segundos números en otra.
Luego, en cada lista **elimina los valores repetidos** para quedarse solo con los distintos.
Después **une ambas listas** en una sola (sin eliminar duplicados entre ellas)
y finalmente **muestra todos los valores** de esa unión.


In [8]:
r1 = rdd.map(lambda t: t[0])
r2 = rdd.map(lambda t: t[1])
r1 = r1.distinct()
r2 = r2.distinct()
res = r2.union(r1)
print(res.collect())

[12, 32, 21, 45, 97, 53, 34, 18, 10, 15, 91, 35, 32, 12, 36, 4, 21, 34, 10, 14, 30, 23, 3, 15, 31, 19]
